In [1]:
import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix
)

# Load dataset
data = pd.read_csv("online_shoppers_intention.csv")
print(data.head())
print(data.shape)

# Separate input and output
X = data.drop("Revenue", axis=1)
y = data["Revenue"].astype(str).str.lower().map({
    "true": 1, "false": 0, "1": 1, "0": 0
})

if y.isna().any():
    raise ValueError("Check the Revenue column")

y = y.astype(int)

# Categorical columns
cat_cols = [
    "Month", "OperatingSystems", "Browser",
    "Region", "TrafficType", "VisitorType", "Weekend"
]

num_cols = [col for col in X.columns if col not in cat_cols]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Save test data
test_data = X_test.copy()
test_data["Revenue"] = y_test
test_data.to_csv("test_data.csv", index=False)

# Preprocessing
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ), cat_cols)
])

# Five ML models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42)
}

# Train and evaluate each model
results = []
os.makedirs("models", exist_ok=True)

for name, algorithm in models.items():

    model = Pipeline([
        ("preprocess", preprocess),
        ("classifier", algorithm)
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "MCC": mcc
    })

    print("\nModel:", name)
    print("Accuracy:", accuracy)
    print("AUC:", auc)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("MCC:", mcc)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    filename = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(model, os.path.join("models", filename))

# Model comparison table
comparison = pd.DataFrame(results)

print("\nComparison of all models")
print(comparison.round(4))

comparison.to_csv("metrics.csv", index=False)

# Best model based on F1 score
best = comparison.loc[comparison["F1"].idxmax(), "Model"]

print("\nBest model based on F1:", best)


   Administrative  Administrative_Duration  Informational  \
0               0                      0.0              0   
1               0                      0.0              0   
2               0                      0.0              0   
3               0                      0.0              0   
4               0                      0.0              0   

   Informational_Duration  ProductRelated  ProductRelated_Duration  \
0                     0.0               1                 0.000000   
1                     0.0               2                64.000000   
2                     0.0               1                 0.000000   
3                     0.0               2                 2.666667   
4                     0.0              10               627.500000   

   BounceRates  ExitRates  PageValues  SpecialDay Month  OperatingSystems  \
0         0.20       0.20         0.0         0.0   Feb                 1   
1         0.00       0.10         0.0         0.0   Feb   

C:\Users\adith\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Model: SVM
Accuracy: 0.8872668288726683
AUC: 0.8824363638190753
Precision: 0.7047244094488189
Recall: 0.468586387434555
F1 Score: 0.5628930817610063
MCC: 0.514938697045181
Confusion Matrix:
[[2009   75]
 [ 203  179]]

Comparison of all models
                 Model  Accuracy     AUC  Precision  Recall      F1     MCC
0  Logistic Regression    0.8808  0.8828     0.7366  0.3586  0.4824  0.4592
1        Decision Tree    0.8585  0.7356     0.5420  0.5576  0.5497  0.4658
2                  KNN    0.8767  0.7633     0.6822  0.3822  0.4899  0.4493
3          Naive Bayes    0.2729  0.7334     0.1726  0.9738  0.2933  0.1289
4                  SVM    0.8873  0.8824     0.7047  0.4686  0.5629  0.5149

Best model based on F1: SVM
